# 10. Discrete Drifting for Peptides

Drifting needs a metric space. Sequences do not have one: Hamming distance between two peptides says almost nothing
about whether they do the same thing, and there is no meaningful "average" of two amino acids. So discrete drifting
is always drifting *somewhere else* — in a learned latent — with a decoder to come back.

*Pepti-drift* (`THEORY.md` §13.1) does exactly that, and in doing so makes a second, larger departure: **its drift is
not anti-symmetric**. Attraction points at an antigen-matched positive; repulsion points at a **fixed pool of toxic
peptides**, not at the model's own samples. The equilibrium is therefore not $q=p$. It is "close to binders, far from
toxins" — a trade-off surface with a knob, $\alpha_-$, that selects a point on it.

That makes this notebook the odd one out, and worth reading carefully for exactly that reason:

* Everything the earlier notebooks proved about equilibrium **does not apply**. There is no $q=p$ fixed point, no
  identifiability argument, no guarantee of distribution matching. This is **objective steering**.
* What *does* carry over is the machinery: kernels, hard negatives, normalisation, warm-up schedules.
* And a failure the paper's formulation is exposed to and we will measure directly: with no self-repulsion, nothing
  in the objective prevents **mode collapse**. We will watch it happen and then fix it by putting the model's own
  samples back into the negative pool — a two-line change that reconciles the two views.

The task is synthetic so that every claim is checkable: a motif grammar where "binds antigen $a$" and "is toxic" are
defined by explicit substrings, so we can count them exactly instead of trusting a classifier.

In [ ]:
import math, time, itertools
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. A motif grammar

Peptides of length 12, built from **four slots** of three residues each. There is a library of 16 possible
three-residue **blocks**: two *binder* blocks per antigen (8 blocks, 4 antigens), two *toxic* blocks, and six neutral
fillers. A peptide binds antigen $a$ if one of its slots holds one of $a$'s binder blocks, and is toxic if any slot
holds a toxic block.

Two design choices matter:

* **The grammar is compositional**, so a compact latent can actually encode a peptide and a decoder can recover it.
  If we used 16 unconstrained residues the sequence would carry ~69 bits of mostly-random content, no 64-dimensional
  latent could hold it, and every result would be limited by the autoencoder rather than by the drift. Check the
  round-trip accuracy in §3 before believing anything downstream — that check is the point of this design.
* **In the training data, binders are toxic 40% of the time.** If the two properties were disjoint the model could
  satisfy both by ignoring one. The overlap is what makes $\alpha_-$ a genuine trade-off, and it is why the method
  needs a warm-up schedule.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
V, N_SLOT, BL = len(AA), 4, 3
L, N_ANT = N_SLOT * BL, 4

_g = torch.Generator().manual_seed(7)
LIB = torch.stack([torch.randperm(V, generator=_g)[:BL] for _ in range(16)]).to(device)
BINDER = {a: [2 * a, 2 * a + 1] for a in range(N_ANT)}     # blocks 0..7
TOXIC = [8, 9]
FILLER = list(range(10, 16))                               # 6 neutral blocks

def blocks_to_seq(b): return LIB[b].reshape(b.shape[0], -1)

def has_block(seqs, k):
    """[B, L] -> [B] bool: does any slot hold block k?"""
    return (seqs.unfold(1, BL, BL) == LIB[k]).all(-1).any(-1)

def is_binder(seqs, a): return torch.stack([has_block(seqs, k) for k in BINDER[a]]).any(0)
def is_toxic(seqs):     return torch.stack([has_block(seqs, k) for k in TOXIC]).any(0)

def make_dataset(n_per=6000, p_toxic_given_binder=0.4, seed=0):
    g = torch.Generator().manual_seed(seed)
    S, Lb = [], []
    fill = torch.tensor(FILLER)
    for a in range(N_ANT):
        b = fill[torch.randint(0, len(FILLER), (n_per, N_SLOT), generator=g)]
        slot = torch.randint(0, N_SLOT, (n_per,), generator=g)
        b[torch.arange(n_per), slot] = torch.tensor(BINDER[a])[
            torch.randint(0, 2, (n_per,), generator=g)]
        tox = torch.rand(n_per, generator=g) < p_toxic_given_binder
        slot2 = (slot + 1 + torch.randint(0, N_SLOT - 1, (n_per,), generator=g)) % N_SLOT
        bt = torch.tensor(TOXIC)[torch.randint(0, 2, (n_per,), generator=g)]
        cur = b[torch.arange(n_per), slot2]
        b[torch.arange(n_per), slot2] = torch.where(tox, bt, cur)
        S.append(blocks_to_seq(b.to(device))); Lb.append(torch.full((n_per,), a))
    return torch.cat(S), torch.cat(Lb).to(device)

def make_toxic_pool(n=4000, seed=1):
    """The repulsion target: toxic peptides carrying no binder block."""
    g = torch.Generator().manual_seed(seed)
    b = torch.tensor(FILLER)[torch.randint(0, len(FILLER), (n, N_SLOT), generator=g)]
    slot = torch.randint(0, N_SLOT, (n,), generator=g)
    b[torch.arange(n), slot] = torch.tensor(TOXIC)[torch.randint(0, 2, (n,), generator=g)]
    return blocks_to_seq(b.to(device))

SEQ, LAB = make_dataset()
TOXPOOL = make_toxic_pool()

def uniq_frac(s): return len({tuple(r) for r in s.cpu().tolist()}) / s.shape[0]
def pos_entropy(s):
    p = torch.stack([(s == v).float().mean(0) for v in range(V)])
    return (-(p.clamp_min(1e-9) * p.clamp_min(1e-9).log()).sum(0) / math.log(V)).mean().item()

print(f"dataset: {SEQ.shape[0]} peptides, {N_ANT} antigens, {len(FILLER)} filler blocks")
for a in range(N_ANT):
    m = LAB == a
    print(f"  antigen {a}: binder rate {is_binder(SEQ[m], a).float().mean():.3f}   "
          f"toxic rate {is_toxic(SEQ[m]).float().mean():.3f}")
print(f"toxic pool: {TOXPOOL.shape[0]} peptides, toxic {is_toxic(TOXPOOL).float().mean():.3f}, "
      f"binder (any antigen) "
      f"{torch.stack([is_binder(TOXPOOL, a) for a in range(N_ANT)]).any(0).float().mean():.3f}")
print(f"data diversity: unique fraction {uniq_frac(SEQ[:4000]):.3f}, "
      f"per-position entropy {pos_entropy(SEQ[:4000]):.3f}")

## 2. The latent space: a frozen encoder on the sphere

Pepti-drift uses a frozen protein language model as $e_P$ and a small trainable head $\phi_P$, then normalises:

$$u_\theta(y) = \mathrm{norm}\big(\phi_P(e_P(y))\big) \in \mathbb{S}^{d_z-1}.$$

We stand in for the PLM with a small bidirectional GRU trained by masked-token prediction on the peptide pool — the
same idea at 1/10⁶ the scale. What matters for the drift is only that the encoder produces a space where "similar
function" means "nearby", which a masked-LM objective gives us because the motifs are the predictable structure in
the data.

The $\ell_2$ normalisation is doing the work Notebook 7 §4 described: on the sphere the kernel becomes a function of
cosine similarity, $\tau$ becomes a dimensionless similarity threshold, and there is no radial degree of freedom for
the field to get wrong.

In [ ]:
D_Z = 64

class SeqEncoder(nn.Module):
    def __init__(self, d=128, d_z=D_Z):
        super().__init__()
        self.emb = nn.Embedding(V + 1, d)                     # index V = [MASK]
        self.rnn = nn.GRU(d, d, num_layers=2, batch_first=True, bidirectional=True)
        self.mlm = nn.Linear(2 * d, V)                        # masked-token head
        self.proj = nn.Sequential(nn.Linear(2 * d, 2 * d), nn.SiLU(), nn.Linear(2 * d, d_z))
    def hidden(self, s): return self.rnn(self.emb(s))[0]       # [B, L, 2d]
    def forward(self, s):                                      # -> unit-norm latent
        return F.normalize(self.proj(self.hidden(s).mean(1)), dim=-1)

def train_encoder(steps=2500, N=256, mask_p=0.25, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    enc = SeqEncoder().to(device)
    opt = torch.optim.AdamW(enc.parameters(), lr)
    pool = torch.cat([SEQ, TOXPOOL])
    for s in range(steps):
        y = pool[torch.randint(0, pool.shape[0], (N,), device=device)]
        m = torch.rand(N, L, device=device) < mask_p
        yin = y.masked_fill(m, V)
        logits = enc.mlm(enc.hidden(yin))
        loss = F.cross_entropy(logits[m], y[m])
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 800 == 0: print(f"  step {s:5d}  masked-LM loss {loss.item():.4f}  (chance {math.log(V):.3f})")
    return enc.eval().requires_grad_(False)

t0 = time.time(); ENC = train_encoder(); print(f"encoder trained in {time.time()-t0:.0f}s")

# does the latent space separate the properties we care about?
with torch.no_grad():
    u_bind = ENC(SEQ[(LAB == 0) & ~is_toxic(SEQ)][:512])
    u_tox = ENC(TOXPOOL[:512])
    u_other = ENC(SEQ[LAB == 1][:512])
print(f"mean cosine similarity, antigen-0 binders vs themselves : "
      f"{(u_bind @ u_bind.T).mean():.3f}")
print(f"                        antigen-0 binders vs antigen-1  : {(u_bind @ u_other.T).mean():.3f}")
print(f"                        antigen-0 binders vs toxins     : {(u_bind @ u_tox.T).mean():.3f}")

## 3. Generator, drift head, and a non-autoregressive decoder

The Pepti-drift pipeline (`THEORY.md` eqs. 57–59), in one forward pass each:

$$z_0 = G_\theta(h_\theta(a),\xi), \qquad z_1 = \mathrm{norm}\big(z_0 + \eta\,D_\theta(z_0, h_\theta(a))\big),
\qquad \hat y = \arg\max \mathrm{Dec}(z_1).$$

The decoder emits all $L$ positions in parallel — no autoregression, so sampling stays 1 NFE end to end. We train it
as an autoencoder head on real peptides ($\mathrm{Dec}(u(y))\to y$), which is what makes the latent sphere decodable
at all.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, d_z=D_Z, h=512):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_z, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, L * V))
    def forward(self, z): return self.net(z).view(-1, L, V)

def train_decoder(steps=3000, N=256, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    dec = Decoder().to(device)
    opt = torch.optim.AdamW(dec.parameters(), lr)
    pool = torch.cat([SEQ, TOXPOOL])
    for s in range(steps):
        y = pool[torch.randint(0, pool.shape[0], (N,), device=device)]
        with torch.no_grad(): u = ENC(y)
        loss = F.cross_entropy(dec(u).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 1000 == 0: print(f"  step {s:5d}  reconstruction CE {loss.item():.4f}")
    return dec

t0 = time.time(); DEC = train_decoder(); print(f"decoder trained in {time.time()-t0:.0f}s")

with torch.no_grad():
    idx = torch.randint(0, SEQ.shape[0], (4096,), device=device)
    y, ya = SEQ[idx], LAB[idx]
    rec = DEC(ENC(y)).argmax(-1)
bind_in = torch.stack([is_binder(y[ya == a], a).float().mean() for a in range(N_ANT)]).mean()
bind_out = torch.stack([is_binder(rec[ya == a], a).float().mean() for a in range(N_ANT)]).mean()
print(f"round-trip token accuracy   : {(rec == y).float().mean():.3f}")
print(f"round-trip binder rate      : {bind_in:.3f} in -> {bind_out:.3f} out")
print(f"round-trip toxic  rate      : {is_toxic(y).float().mean():.3f} in -> "
      f"{is_toxic(rec).float().mean():.3f} out")

The round-trip accuracy should be at or near 1.0 — the compositional grammar was chosen so that it can be. That
matters because it is the **ceiling** on everything that follows: whatever the drift achieves in latent space, the
decoder has to survive it, and a mediocre autoencoder would silently cap every number in this notebook. This is the discrete analogue of Notebook 7 §3's warning — an equilibrium in latent space
is not an equilibrium in sequence space, and here the gap is bigger, because $\arg\max$ is a projection that throws
away everything the latent knew about *how confident* it was.

## 4. The drift: attraction to a teacher, repulsion from a region

$$V^{+}_{\text{att}}(\bar x) = \alpha_{+}\big(u^{+} - \bar x\big), \qquad
V^{-}_{\text{rep}}(\bar x) = \alpha_{-}(e)\sum_{k} w^{-}_{k}\big(\bar x - u^{-}_{k}\big),$$

with $w^{-}$ a softmax over distances to the $K$ **hardest** negatives — the toxins nearest this sample in latent
space — and the target $\tilde x = \mathtt{sg}[\mathrm{norm}(\bar x + V)]$.

Three departures from everything before, worth naming:

1. **The positive is a single antigen-matched teacher**, not a distribution — the single-positive regime of
   Notebook 8 §4, with the same consequence: no kernel-based structure in the attraction.
2. **The negatives are a fixed external pool**, so repulsion carries no information about where the model's own mass
   is. It cannot prevent collapse; it can only prevent *toxicity*.
3. **The two are separately weighted** by $\alpha_+,\alpha_-$, which is precisely the anti-symmetry violation
   Notebook 1 §9 showed to be fatal for distribution matching. Here it is deliberate: this objective was never trying
   to match a distribution.

In [ ]:
class AntigenEmb(nn.Module):
    def __init__(self, d=64):
        super().__init__(); self.emb = nn.Embedding(N_ANT, d)
    def forward(self, a): return self.emb(a)

class GenDrift(nn.Module):
    """z0 = G(h(a), xi);  z1 = norm(z0 + eta * D(z0, h(a)))"""
    def __init__(self, d_z=D_Z, d_a=64, d_xi=32, h=512):
        super().__init__()
        self.d_xi = d_xi
        self.ant = AntigenEmb(d_a)
        self.G = nn.Sequential(nn.Linear(d_a + d_xi, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                               nn.Linear(h, d_z))
        self.D = nn.Sequential(nn.Linear(d_z + d_a, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                               nn.Linear(h, d_z))
    def forward(self, a, xi=None, eta=1.0):
        ha = self.ant(a)
        if xi is None: xi = torch.randn(a.shape[0], self.d_xi, device=a.device)
        z0 = F.normalize(self.G(torch.cat([ha, xi], -1)), dim=-1)
        delta = self.D(torch.cat([z0, ha], -1))
        return F.normalize(z0 + eta * delta, dim=-1), delta

def alpha_minus(epoch, final, e_start=300, e_warm=600):
    """Pepti-drift eq. 60: ramp the repulsion in only after a stable attraction phase."""
    return final * min(1.0, max(0.0, epoch - e_start) / e_warm)

def hardest_negatives(x, pool_u, K):
    """The K toxins closest to each sample in latent space, and their softmax weights."""
    sim = x @ pool_u.T                                     # cosine, since everything is unit norm
    top = sim.topk(K, dim=-1)
    w = top.values.softmax(-1)                             # nearest gets the most weight
    return pool_u[top.indices], w                          # [N,K,d], [N,K]

Now the training loop. It carries two knobs that are **not** in the published formulation, both introduced here
because the published one degenerates on this task and we would rather show that than hide it:

* **`pos_mode`** — `"random"` draws a uniformly random antigen-matched positive as the teacher, as described.
  `"nearest"` uses a kernel-weighted positive (a soft nearest neighbour among a pool of antigen-matched positives),
  i.e. the ordinary drifting attraction of Notebook 1. §5 shows why: with a *random* teacher, an MSE regression
  converges to $\mathbb{E}[u^{+}\mid a]$, the per-antigen **mean** latent, so the model provably collapses to one
  sequence per antigen no matter what else you do.
* **`self_repel`** — puts the model's own samples back into the negative pool. Off reproduces the published field;
  on restores Notebook 1's anti-collapse mechanism. §7.

Watch the signs. The drift is $V = V^{+}_{\text{att}} + V^{-}_{\text{rep}}$ where $V^{-}_{\text{rep}}$ is written in
the *push-away* form $\sum_k w_k(\bar x - u^{-}_k)$ — a vector pointing from the negative toward the sample, so it is
**added**. Written in the mean-shift form of Notebook 1 the same term is subtracted. Getting this backwards attracts
samples to the toxins, trains perfectly stably, and produces a model that generates exactly what you were trying to
avoid.

In [ ]:
@torch.no_grad()
def encode_pool():
    U_tox = torch.cat([ENC(TOXPOOL[i:i+1024]) for i in range(0, TOXPOOL.shape[0], 1024)])
    U_pos = torch.cat([ENC(SEQ[i:i+1024]) for i in range(0, SEQ.shape[0], 1024)])
    return U_tox, U_pos
U_TOX, U_POS = encode_pool()
POS_BY_ANT = [torch.where(LAB == a)[0] for a in range(N_ANT)]

def train_peptide(alpha_pos=0.3, alpha_neg_final=0.0, K=8, steps=2500, N=256, lr=1e-3,
                  pos_mode="nearest", tau_pos=0.05, P=128, warmup=True, lam_disp=0.1,
                  self_repel=1.0, tau_self=0.1, seed=0):
    torch.manual_seed(seed)
    g = GenDrift().to(device)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95))
    for s in range(steps):
        a = torch.randint(0, N_ANT, (N,), device=device)
        x, delta = g(a)
        an = alpha_minus(s, alpha_neg_final) if warmup else alpha_neg_final
        with torch.no_grad():
            # --- the antigen-matched positive teacher
            if pos_mode == "random":
                idx = torch.stack([POS_BY_ANT[int(c)][torch.randint(
                    0, len(POS_BY_ANT[int(c)]), (1,), device=device)][0] for c in a])
                u_pos = U_POS[idx]
            else:
                u_pos = torch.zeros_like(x)
                for c in range(N_ANT):                       # kernel-weighted, per antigen
                    m = a == c
                    if m.sum() == 0: continue
                    pool = U_POS[POS_BY_ANT[c][torch.randint(0, len(POS_BY_ANT[c]), (P,), device=device)]]
                    w = ((x[m] @ pool.T) / tau_pos).softmax(-1)
                    u_pos[m] = F.normalize(w @ pool, dim=-1)
            V = alpha_pos * (u_pos - x)                        # attraction
            if an > 0:                                        # repulsion from the K hardest toxins
                u_neg, w = hardest_negatives(x, U_TOX, K)
                V = V + an * (w[..., None] * (x[:, None] - u_neg)).sum(1)
            if self_repel > 0:                                # repulsion from our own samples
                sim = x @ x.T - torch.eye(N, device=device) * 1e6
                ws = (sim / tau_self).softmax(-1)
                V = V + self_repel * (x - ws @ x)
            target = F.normalize(x + V, dim=-1)
        loss = F.mse_loss(x, target) + lam_disp * delta.pow(2).mean()
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
    return g

@torch.no_grad()
def evaluate(g, n=4000, tag=""):
    a = torch.arange(N_ANT, device=device).repeat_interleave(n // N_ANT)
    seqs = DEC(g(a)[0]).argmax(-1)
    bind = torch.stack([is_binder(seqs[a == c], c).float().mean() for c in range(N_ANT)]).mean()
    tox = is_toxic(seqs).float().mean()
    uniq, ent = uniq_frac(seqs), pos_entropy(seqs)
    if tag:
        print(f"{tag:36s} binder {bind:.3f}   toxic {tox:.3f}   unique {uniq:.3f}   entropy {ent:.3f}")
    return dict(binder=bind.item(), toxic=tox.item(), unique=uniq, entropy=ent)

print(f"{'training data (reference)':36s} binder {1.000:.3f}   "
      f"toxic {is_toxic(SEQ).float().mean():.3f}   "
      f"unique {uniq_frac(SEQ[:4000]):.3f}   entropy {pos_entropy(SEQ[:4000]):.3f}")

## 5. First, the two degeneracies

Before measuring anything about toxicity, establish what the published field does on this task, because otherwise
every later number is measured on a collapsed model.

In [ ]:
print("(a) random teacher, no self-repulsion  -- the published formulation:")
evaluate(train_peptide(pos_mode="random", self_repel=0.0), tag="    alpha_-=0")
evaluate(train_peptide(pos_mode="random", self_repel=0.0, alpha_neg_final=1.0), tag="    alpha_-=1")

print("\n(b) kernel-weighted teacher, still no self-repulsion:")
evaluate(train_peptide(pos_mode="nearest", self_repel=0.0), tag="    alpha_-=0")
evaluate(train_peptide(pos_mode="nearest", self_repel=0.0, alpha_neg_final=1.0), tag="    alpha_-=1")

print("\n(c) kernel-weighted teacher + self-repulsion:")
evaluate(train_peptide(pos_mode="nearest", self_repel=1.0), tag="    alpha_-=0")
evaluate(train_peptide(pos_mode="nearest", self_repel=1.0, alpha_neg_final=1.0), tag="    alpha_-=1")

Look at the `unique` column. In (a) and (b) it is $\sim0.002$ — **about eight distinct peptides out of 4000**, which
is one or two per antigen. The model has collapsed completely, and the binder and toxic rates are then just
properties of those few sequences, not of a distribution. Only (c) produces anything you could call a generative
model.

Two distinct mechanisms are at work.

**The random teacher collapses provably.** With $u^{+}$ drawn uniformly from the antigen's positives and an MSE
regression onto $\mathtt{sg}[\mathrm{norm}(\bar x + \alpha_+(u^{+}-\bar x))]$, the minimiser over a fixed antigen is
the conditional mean $\mathbb{E}[u^{+}\mid a]$. There is one such point per antigen, so the model has no reason to
use $\xi$ at all. The kernel-weighted teacher of (b) fixes this in principle — each sample is pulled toward
*whichever* positive it is nearest, so different $\xi$ give different targets — but only if the samples are already
spread out.

**And they are not, because there is nothing to spread them.** At initialisation every sample sits in nearly the
same place, so they all have the same nearest positive, so they all move to the same place. The collapse is
self-reinforcing, and (b) shows the kernel teacher alone cannot escape it.

**Self-repulsion is what breaks the symmetry.** In (c) the samples repel each other from step 0, they spread, they
find different nearest positives, and the kernel teacher can finally do its job. Unique fraction goes from 0.002 to
above 0.9.

This is the notebook's central observation, and it is a criticism of the formulation rather than of the paper's
results — Pepti-drift reports sequence-diversity metrics, so their real system evidently avoids this, presumably
through a richer antigen conditioning and a larger effective positive set. But the *field as written* has no
anti-collapse term, and on a task where you can count the collapse, it collapses.

## 6. Does repulsion actually reduce toxicity?

Now that the model is not collapsed, the question is meaningful. Sweep $\alpha_-$ with self-repulsion held fixed.

In [ ]:
res = {}
for an in [0.0, 0.25, 0.5, 1.0, 2.0]:
    res[an] = evaluate(train_peptide(alpha_neg_final=an, self_repel=1.0), tag=f"alpha_- = {an}")
print(f"\ntoxicity: {res[0.0]['toxic']:.3f} -> {res[1.0]['toxic']:.3f}   "
      f"({res[0.0]['toxic']/max(res[1.0]['toxic'],1e-9):.1f}x reduction)")
print(f"binder  : {res[0.0]['binder']:.3f} -> {res[1.0]['binder']:.3f}")
print(f"unique  : {res[0.0]['unique']:.3f} -> {res[1.0]['unique']:.3f}")

ans = sorted(res)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ans, [res[a]["toxic"] for a in ans], "o-", label="toxic rate")
ax[0].plot(ans, [res[a]["binder"] for a in ans], "s-", label="binder rate")
ax[0].plot(ans, [res[a]["unique"] for a in ans], "^-", label="unique fraction")
ax[0].axhline(is_toxic(SEQ).float().mean().item(), ls="--", c="0.5", lw=1, label="data toxic rate")
ax[0].set_xlabel(r"$\alpha_-$"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot([res[a]["toxic"] for a in ans], [res[a]["binder"] for a in ans], "o-")
for a in ans:
    ax[1].annotate(f"{a}", (float(res[a]["toxic"]), float(res[a]["binder"])), fontsize=7)
ax[1].set_xlabel("toxic rate"); ax[1].set_ylabel("binder rate")
ax[1].set_title("the trade-off surface", fontsize=9); ax[1].grid(alpha=0.3)
plt.tight_layout()

Repulsion works, and it does not cost binding: from $\alpha_-=0$ to $\alpha_-=1$ the toxic rate falls roughly
threefold while the binder rate *rises*. That is not a coincidence — the toxic and binder blocks compete for slots,
so pushing away from the toxic region also pushes toward peptides whose slots hold binder blocks instead.

But read the `unique` column alongside. Diversity is falling the whole time, and at $\alpha_-=2$ the model collapses
again: toxicity 0.000, and one sequence per antigen. **A toxicity of zero achieved by emitting a single peptide is
not a result.** This is why the right-hand panel is the honest way to report an objective-steering method: there is
no correct $\alpha_-$, there is a curve, and a third axis (diversity) that a two-axis plot hides.

## 7. The warm-up schedule

$$\alpha_{-}(e) = \alpha_{-}^{\text{final}}\cdot\min\!\Big(1,\ \frac{\max(0,\,e-e_{\text{start}})}{e_{\text{warmup}}}\Big)$$

The stated rationale is that binders and toxins overlap in the data, so attraction and repulsion pull in partly
opposite directions from step 0; ramping $\alpha_-$ in later lets a stable binding-oriented trajectory form first.
Test it.

In [ ]:
for an in [1.0, 2.0]:
    for w in [True, False]:
        evaluate(train_peptide(alpha_neg_final=an, self_repel=1.0, warmup=w),
                 tag=f"alpha_-={an}, warmup={'on ' if w else 'off'}")

**A null result, reported as one.** On this task the warm-up makes no measurable difference: with and without it the
numbers agree to within seed noise, and at $\alpha_-=2$ both collapse.

That is worth understanding rather than filing away. The warm-up matters when the two objectives genuinely fight
*early*, which requires attraction and repulsion to be comparable in magnitude in the region the model starts in.
Here the toxic pool sits at cosine similarity ~0.38 from the binders (§2) — far enough that the hardest-negative
repulsion is weak at initialisation and only becomes significant once the model has already moved toward the
binders. The schedule is, in effect, already implicit in the geometry.

The prediction this makes: warm-up should matter when the toxic and binder regions overlap more, i.e. when
`p_toxic_given_binder` is raised toward 1 or the encoder separates the two classes less. That is a one-line
experiment and it is the right way to settle whether a schedule is load-bearing or cargo.

The displacement penalty $\lambda\|\Delta\|^2$ plays a related and more clearly useful role: it stops the drift head
from making one enormous jump that satisfies repulsion by leaving the useful region of the sphere entirely.

## 8. The diversity/affinity axis

Self-repulsion turned out to be necessary (§5). It is also a knob, and it moves the model along a *different* axis
from $\alpha_-$.

In [ ]:
for sr in [0.0, 0.5, 1.0, 2.0, 4.0]:
    evaluate(train_peptide(alpha_neg_final=1.0, self_repel=sr), tag=f"self_repel = {sr}")

Two regimes. Below a threshold (here $\approx1$) self-repulsion is simply too weak to break the collapse, and the
`unique` column stays at $\sim0.002$ — the binder rate of 0.47 at `self_repel = 0.5` is the binder rate of *three
sequences*, not of a distribution, and should not be read as the best row. Above the threshold there is a clean
monotone trade-off: more self-repulsion buys unique-sequence fraction and per-position entropy and costs binder rate.

Compare with §6, where $\alpha_-$ bought safety and cost diversity. The two knobs are close to orthogonal:

* $\alpha_-$ moves you along **safety vs affinity**;
* self-repulsion moves you along **diversity vs affinity**.

This is the reconciliation between this notebook and all the earlier ones. Pepti-drift's field is the *steering* half
of drifting; the anti-symmetric self-repulsion is the *distribution-matching* half. They compose, because they are
just different terms in the same negative pool — exactly the freedom Notebook 5 §3 identified when it wrote
classifier-free guidance as a change to the negative distribution. In that language:

$$\text{negatives} \;=\; \underbrace{q_\theta}_{\text{diversity}} \;\cup\; \underbrace{p_{\text{toxic}}}_{\text{safety}}
\;\cup\; \underbrace{p_{\text{data}}(\cdot|\varnothing)}_{\text{CFG}}$$

with a weight on each. **Any property you can sample a repulsive set for can be added here** — off-target binders,
aggregation-prone sequences, known immunogenic motifs — and the cost is one more term in a softmax.

## 9. What is different about discrete data

Collecting the hazards, because they are not obvious from the continuous case:

1. **There is no metric until you build one.** The entire method rests on the encoder, and unlike images there is no
   off-the-shelf perceptual space — protein language models are the closest thing and they are trained for a
   different objective. If two sequences with the same function are far apart in your latent, drifting is optimising
   the wrong thing and nothing downstream will tell you.
2. **$\arg\max$ is a lossy projection.** Latent equilibrium does not imply sequence-space equilibrium. Measure in
   sequence space; the round-trip accuracy of §3 is the ceiling, and the gap between latent and sequence metrics is a
   real quantity worth watching.
3. **Averages of latents are not latents of averages.** A kernel-weighted mean of two binder latents may decode to a
   peptide with neither motif. Normalising onto the sphere mitigates this but does not remove it; it is the discrete
   cousin of the manifold-violation problem in Notebook 9 §6, and another argument for weight-space rather than
   direction-space modifications.
4. **Hard negatives matter more than in the continuous case**, because the undesirable region is small and a random
   draw from it is usually far away and carries almost no gradient. Same issue as Notebook 6 §3, solved here by
   mining rather than by normalising.
5. **Discreteness hides collapse.** A model emitting eight distinct peptides can still show respectable per-position
   entropy if those eight differ everywhere — as §5 shows, where entropy stayed at 0.3 while the unique fraction was
   0.002. **Report unique-sequence fraction and entropy together, never one alone.**

## Summary

* Discrete drifting = drift in an $\ell_2$-normalised learned latent, decode non-autoregressively. Sampling is one
  forward pass; the round-trip accuracy of the autoencoder is the ceiling on everything else.
* Pepti-drift's field is **deliberately not anti-symmetric**: attraction to an antigen-matched teacher, repulsion
  from a fixed toxic pool. The equilibrium is a trade-off point, not $q=p$, and the honest report is a curve.
* **As written, the field has no anti-collapse term, and it collapses**: ~8 distinct peptides out of 4000. A random
  teacher collapses provably (the MSE minimiser is the per-antigen mean); a kernel teacher collapses in practice
  because nothing spreads the samples out in the first place.
* Adding the model's own samples to the negative pool fixes it, and then the toxicity result is real: a ~3$\times$
  reduction in toxic-motif frequency with the binder rate *rising*.
* The warm-up schedule made **no measurable difference here** — a null result, with a stated prediction for when it
  would matter.
* Two near-orthogonal knobs: $\alpha_-$ (safety vs affinity) and self-repulsion (diversity vs affinity). The negative
  pool is a composition, and anything you can sample can join it.

---

That closes the method. `THEORY.md` has the full derivations and the open problems; `PHYSICS.md` reads the same
mathematics as an interacting-particle system; and `PROTENIX_DRIFTING.md` applies all of it to one concrete
engineering question.

Next: [11. From Notebooks to picodrift](<11. From Notebooks to picodrift.ipynb>) — the prerequisites for the production
application, where drifting meets real protein structure.